In [1]:
import pandas as pd
import re
import sys
import time

In [6]:
nombre = "Now"
ruta_base = "/home/jrodarte/Proyectos/prestadero/documentos"

ruta_original = f"{ruta_base}/movimientos{nombre}.csv"
ruta_limpia = f"{ruta_base}/movimientos{nombre}Cln.csv"

df = pd.read_csv(ruta_original)

df = df.replace(r'\n\s+', '', regex=True)
df = df.replace(r'\n\s+0', ' 0', regex=True)

dfMovimientos = df

#dfMovimientos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("documentos/movimientosNow.csv")


In [7]:
dfDetalleMovimiento = dfMovimientos[['Autorización', 'Detalle']].copy()
dfDetalleMovimiento.columns = ['autorizacion', 'detalle']

dfMovimientos = dfMovimientos[
    [
        'Autorización',
        'Fecha operación',
        'Tipo',
        'Movimiento',
        'Importe',
        'Estatus',
        'Ref. 1'
    ]
]

dfMovimientos.columns = [
    'autorizacion',
    'feoperacion',
    'tipo',
    'movimiento',
    'importe',
    'estatus',
    'referencia'
]

# -------------------------
# FECHAS
# -------------------------
dfMovimientos['feoperacion'] = pd.to_datetime(
    dfMovimientos['feoperacion'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
)

dfMovimientos['femovimiento'] = dfMovimientos['feoperacion'].dt.date

dfMovimientos['feoperacion_mes'] = (
    dfMovimientos['feoperacion']
    .dt.to_period('M')
    .dt.to_timestamp()
    .dt.date
)

dfMovimientos['feoperacion'] = dfMovimientos['feoperacion_mes']

dfMovimientos.drop(columns=["feoperacion_mes"], inplace=True)

# -------------------------
# LIMPIAR IMPORTE
# -------------------------
dfMovimientos['importe'] = (
    dfMovimientos['importe']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)
    .replace('', '0')
    .astype(float)
)

# -------------------------
# USUARIO SOLICITANTE
# -------------------------
dfMovimientos['nombre_usuario_solicitante'] = (
    dfMovimientos['referencia']
    .where(dfMovimientos['referencia'].notna())
    .str.split(':')
    .str[1]
    .str.strip()
)
dfMovimientos.dtypes

/tmp/ipykernel_176535/897294791.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfMovimientos['feoperacion'] = pd.to_datetime(
/tmp/ipykernel_176535/897294791.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfMovimientos['femovimiento'] = dfMovimientos['feoperacion'].dt.date


autorizacion                    int64
feoperacion                    object
tipo                           object
movimiento                     object
importe                       float64
estatus                        object
referencia                     object
femovimiento                   object
nombre_usuario_solicitante     object
dtype: object

In [6]:
# Datos de conexión
user = "jrodarte"
password = "roma1993_"

dfTipoMovimientos = ss.read.format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/prestaderodb") \
    .option("dbtable", "(select * from prestadero.tipomovimientos)") \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

#dfTipoMovimientos.show()

In [7]:
def limpiarImporte(importe):
    return importe.replace("$", "").replace(",", "")


limpiarImpoUDF = udf(limpiarImporte, StringType())

In [8]:
nuevosNombresMov = ['autorizacion', 'feoperacion', 'tipo', 'movimiento', 'importe', 'estatus', 'referencia']
nuevosNombresDetMov = ['autorizacion', 'detalle']
dfDetalleMovimiento = dfMovimientos.select('Autorización', 'Detalle')
dfMovimientos = dfMovimientos.selectExpr(
    "`Autorización`",
    "`Fecha operación`",
    "`Tipo`",
    "`Movimiento`",
    "`Importe`",
    "`Estatus`",
    "`Ref. 1`"
)

In [27]:
dfMovimientos = dfMovimientos.toDF(*nuevosNombresMov)
dfMovimientos = dfMovimientos.withColumn('feoperacion', to_timestamp(col("feoperacion"), "dd/MM/yyyy HH:mm:ss"))
dfMovimientos = dfMovimientos.withColumn('importeS', limpiarImpoUDF(col("importe")))
dfMovimientos = dfMovimientos.withColumn('importe', col("importeS").cast(DoubleType())).drop("importeS")
dfDetalleMovimiento = dfDetalleMovimiento.toDF(*nuevosNombresDetMov)
dfMovimientos.show(Truncate = False)
dfMovimientos.schema

In [28]:
dfMovimientos = dfMovimientos.join(dfTipoMovimientos, dfMovimientos.movimiento == dfTipoMovimientos.tipomovimiento, "inner")
dfMovimientos = dfMovimientos.drop('tipomovimiento').drop('movimiento')
#dfMovimientos.schema

In [20]:
dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [35]:
conteoMovimientos = dfMovimientos.count()
print(conteoMovimientos)

95


In [34]:
try:
    dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()
except Exception as inst:
    print(type(inst))    # the exception type
    print(inst.args)     # arguments stored in .args
    print(inst)          # __str__ allows args to be printed directly,

<class 'py4j.protocol.Py4JJavaError'>
('An error occurred while calling o317.save.\n', JavaObject id=o318)
An error occurred while calling o317.save.
: org.postgresql.util.PSQLException: Connection to localhost:543 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:346)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:273)
	at org.postgresql.Driver.makeConnection(Driver.java:446)
	at org.postgresql.Driver.connect(Driver.java:298)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:50)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$creat